In [15]:
import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine

#### Conexion con las bases de datos

In [17]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['SOURCE_DB']
    config_etl = config['ETL_PRO']
    
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")

engine_origen = create_engine(url_co)
engine_destino = create_engine(url_etl)

In [18]:
sede = pd.read_sql_table("sede", engine_origen)
ciudad = pd.read_sql_table("ciudad", engine_origen)
departamento = pd.read_sql_table("departamento", engine_origen)

In [19]:
sede.head()

,sede_id,nombre,direccion,telefono,nombre_contacto,ciudad_id,cliente_id
0,10,FARALLONES /123,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
1,11,REMEDIOZ/ 123,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
2,13,DIME / LOS ROJOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
3,14,DESPACHOS / LOS ROJOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
4,23,POPAYAN BODEGA 28 / A,Los angeles distrito Latino,310-70000,JUAN PEREZ,3,11


In [20]:
ciudad.head()

,ciudad_id,nombre,departamento_id
0,6,BUGA,1
1,5,BOGOTA,2
2,4,PASTO,4
3,3,POPAYAN,3
4,2,PALMIRA,1


In [21]:
departamento.head()

,departamento_id,nombre
0,4,NARIÑO
1,3,CAUCA
2,2,CUNDINAMARCA
3,1,VALLE DEL CAUCA


In [22]:
dim_sede = sede.merge(ciudad, left_on='ciudad_id', right_on='ciudad_id', how='left')

In [23]:
dim_sede = dim_sede.merge(departamento, left_on='departamento_id', right_on='departamento_id', how='left')

In [24]:
dim_sede.head()

,sede_id,nombre_x,direccion,telefono,nombre_contacto,ciudad_id,cliente_id,nombre_y,departamento_id,nombre
0,10,FARALLONES /123,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4,CALI,1,VALLE DEL CAUCA
1,11,REMEDIOZ/ 123,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4,CALI,1,VALLE DEL CAUCA
2,13,DIME / LOS ROJOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4,CALI,1,VALLE DEL CAUCA
3,14,DESPACHOS / LOS ROJOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4,CALI,1,VALLE DEL CAUCA
4,23,POPAYAN BODEGA 28 / A,Los angeles distrito Latino,310-70000,JUAN PEREZ,3,11,POPAYAN,3,CAUCA


In [37]:
dim_sede.rename(columns={
    'sede_id': 'id_sede',
    'nombre_x': 'nombre_sede',
    'nombre_y': 'ciudad',
    'nombre': 'departamento'
}, inplace=True)

In [38]:
dim_sede.columns.tolist()

['id_sede',
 'nombre_sede',
 'direccion',
 'telefono',
 'nombre_contacto',
 'ciudad_id',
 'cliente_id',
 'ciudad',
 'departamento_id',
 'departamento']

In [40]:
columnas_necesarias = ["id_sede", "nombre_sede", "ciudad", "departamento", "direccion"]
dim_sede = dim_sede[columnas_necesarias].copy()
dim_sede.head()

,id_sede,nombre_sede,ciudad,departamento,direccion
0,10,FARALLONES /123,CALI,VALLE DEL CAUCA,Los angeles distrito Latino
1,11,REMEDIOZ/ 123,CALI,VALLE DEL CAUCA,Los angeles distrito Latino
2,13,DIME / LOS ROJOS,CALI,VALLE DEL CAUCA,Los angeles distrito Latino
3,14,DESPACHOS / LOS ROJOS,CALI,VALLE DEL CAUCA,Los angeles distrito Latino
4,23,POPAYAN BODEGA 28 / A,POPAYAN,CAUCA,Los angeles distrito Latino


In [42]:
dim_sede.to_sql('dim_sede',con=engine_destino,if_exists='replace',index_label='key_dim_sede')

52